In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle
from collections import defaultdict
from tqdm.notebook import trange, tqdm
from icecream import ic

import pandas as pd
import numpy as np
import scipy
import sklearn
import prince

import seaborn as sns
from matplotlib import pyplot as plt

from plotly import express as px
from plotly import graph_objects as go
import plotly.io as pio

In [ ]:
from rsfp.metrics import *
from rsfp.constants import *
from rsfp.data import *
from rsfp.matching import *
from rsfp.dimensionality_reduction import *
from rsfp.utils import *

In [ ]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)
pd.options.plotting.backend = "plotly"

# Data

## 2023

In [ ]:
# load 2023 datasets
df_voters, df_candidates, df_questions = build_all(verbose=True)

In [ ]:
# load uncleaned 2023 voter dataset
df_voters_all = build_voters(clean=False)
df_voters_all

In [ ]:
# load recommendation data for L2 distance metric
df_voters = df_voters.load_candidate_voting_recommendations('df_voters_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load list recommendations for L2 distance metric
df_voters = df_voters.load_list_voting_recommendations('df_voters_list_recommendations_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load recommendation data for evaluated distance metrics
df_voters = df_voters.load_candidate_voting_recommendations('df_voters_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# load list recommendations for evaluated distance metrics
df_voters = df_voters.load_list_voting_recommendations('df_voters_list_recommendations_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# add dimensionality reduction columns
df_candidates, obj = add_pca_cols(df_candidates, n_dims=2, return_obj=True)
df_voters = add_pca_cols(df_voters, n_dims=2, obj=obj)

df_candidates, obj = add_ca_cols(df_candidates, n_dims=2, return_obj=True)
df_voters = add_ca_cols(df_voters, n_dims=2, obj=obj)

df_candidates, obj = add_tsvd_cols(df_candidates, n_dims=2, return_obj=True)
df_voters = add_tsvd_cols(df_voters, n_dims=2, obj=obj)

In [ ]:
# plot PCA distribution of candidates
fig = px.scatter(df_candidates[~df_candidates['_party'].isin(['Parteilos', 'Andere'])], x='PCA_0', y='-PCA_1', color='_party', color_discrete_map=PARTY2COLOR)
fig.update_layout(
    xaxis_title='Left - Right',
    yaxis_title='Conservative - Liberal',
    legend=dict(
        title='Party',
        x=1,
        y=1,
        xanchor='left',
        yanchor='top',
        itemsizing='constant',
    )
).update_traces(marker_size=3)


## 2019

In [ ]:
# load 2019 datasets
df_voters19, df_candidates19, df_questions19 = build_all19(verbose=True)

In [ ]:
# load uncleaned 2019 voter dataset
df_voters19_all = build_voters19(clean=False)

In [ ]:
# load recommendation data for L2 distance metric
df_voters19 = df_voters19.load_candidate_voting_recommendations('df_voters19_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])
df_candidates19 = df_candidates19.load_candidate_recommendation_counts('df_candidates19_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load list recommendations for L2 distance metric
df_voters19 = df_voters19.load_list_voting_recommendations('df_voters19_list_recommendations_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load recommendation data for evaluated distance metrics
df_voters19 = df_voters19.load_candidate_voting_recommendations('df_voters19_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)
df_candidates19 = df_candidates19.load_candidate_recommendation_counts('df_candidates19_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# load list recommendations for evaluated distance metrics
df_voters19 = df_voters19.load_list_voting_recommendations('df_voters19_list_recommendations_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# add dimensionality reduction columns
df_candidates19, obj = add_pca_cols(df_candidates19, n_dims=2, return_obj=True)
df_voters19 = add_pca_cols(df_voters19, n_dims=2, obj=obj)

df_candidates19, obj = add_ca_cols(df_candidates19, n_dims=2, return_obj=True)
df_voters19 = add_ca_cols(df_voters19, n_dims=2, obj=obj)

df_candidates19, obj = add_tsvd_cols(df_candidates19, n_dims=2, return_obj=True)
df_voters19 = add_tsvd_cols(df_voters19, n_dims=2, obj=obj)

# Distance Methods

## L1 with Bonus Distance

In [ ]:
np.set_printoptions(linewidth=200)

In [ ]:
voter_answers = df_voters.iloc[:1000].a().values
voter_weights = df_voters.iloc[:1000].w().values
candidate_answers = df_candidates.iloc[:100].a().values

In [ ]:
df_questions['type'].value_counts()

In [ ]:
def update_question_type2idx(question_type2idx, nan_mask):
    sliced_columns = np.where(nan_mask)
    updated_question_type2idx = {}
    for category, indices in question_type2idx.items():
        updated_indices = [idx - sum(idx > col for col in sliced_columns) for idx in indices if idx not in sliced_columns]
        updated_question_type2idx[category] = updated_indices
    return updated_question_type2idx

In [ ]:
# Example dictionary mapping categories to column indices
category_to_indices = {4: [0, 1, 2], 5: [3, 4, 5], 7: [6, 7, 8]}

# Assume some columns are sliced out
sliced_columns = [0, 1, 10]  # Example of sliced columns

# Update column indices after slicing
updated_category_to_indices = {}
for category, indices in category_to_indices.items():
    updated_indices = [idx - sum(idx > col for col in sliced_columns) for idx in indices if idx not in sliced_columns]
    updated_category_to_indices[category] = updated_indices

print(updated_category_to_indices)


In [ ]:
{qt: np.where(df_questions19['_n_options'] == qt)[0] for qt in [4, 5, 7]}

In [ ]:
unique_counts = np.apply_along_axis(lambda x: len(np.unique(x)), axis=0, arr=voter_answers)
unique_counts

In [ ]:
np.apply_along_axis(lambda x: len(np.unique(x)), axis=0, arr=np.vstack((voter_answers, candidate_answers)))

In [ ]:
dists = []
for question_type in [4, 5, 7]:
    dist_mat = L1_DIST_MAT[:, ROW_COL_INDICES[question_type]][ROW_COL_INDICES[question_type]]
    dist_mat -= np.diag(dist_mat.sum(axis=0) - np.min(dist_mat.sum(axis=0)))
    ic(dist_mat)
    padded_dist_mat = np.zeros((9, 9)) - 1e6
    padded_dist_mat[np.ix_(ROW_COL_INDICES[question_type], ROW_COL_INDICES[question_type])] = dist_mat
    ic(padded_dist_mat)

    n_unique_options = np.apply_along_axis(lambda x: len(np.unique(x)), axis=0, arr=np.vstack((voter_answers, candidate_answers)))
    question_type_idx = np.where(n_unique_options == question_type)[0]
    if len(question_type_idx) != question_type2count[question_type]:
        warnings.warn(f"Expected {question_type2count[question_type]} questions with {question_type} answer options but found {len(question_type_idx)}.", UserWarning)
    dists.append(calculate_dist_mat_distances(voter_answers[:, question_type_idx], candidate_answers[:, question_type_idx], voter_weights[:, question_type_idx], dist_mat=padded_dist_mat))
dists = np.stack(dists).sum(axis=0)
dists.shape

In [ ]:
dists2 = calculate_distances(voter_answers, candidate_answers, voter_weights, distance_method='L1')

In [ ]:
px.histogram((dists2 - dists.sum(axis=0)).ravel())

In [ ]:
type4_question_idx

In [ ]:
np.where(df_questions['_n_options'] == 4)[0]

In [ ]:
L1_DIST_MAT[: ROW_COL_INDICES[question_type]][ROW_COL_INDICES[question_type]]

In [ ]:
L1_DIST_MAT = np.abs(np.subtract.outer(ANSWER_POSSIBILITIES, ANSWER_POSSIBILITIES)).astype(float)
L1_DIST_MAT

In [ ]:
df_questions = build_questions()

In [ ]:
L1_DIST_MAT[:, RC4][RC4]

## Mahalonobis Distance

In [ ]:
candidate_answers = df_candidates.a().values

In [ ]:
covariance_matrix = np.cov(candidate_answers, rowvar=False)

# Calculate the inverse of the covariance matrix
inverse_covariance_matrix = np.linalg.inv(covariance_matrix)

In [ ]:
np.linalg.inv(cov)

In [ ]:
cov = [
    [1, 0], 
    [0, 10]
]
samples = np.random.multivariate_normal(np.zeros(len(cov)), cov=cov, size=100)

In [ ]:
cov = [
    [1, 0], 
    [0, 1]
]
samples2 = np.random.multivariate_normal(np.zeros(len(cov)), cov=cov, size=100)

In [ ]:
voter_answers = df_voters.iloc[:10000].a().values
candidate_answers = df_candidates.iloc[:100].a().values
cov = np.cov(candidate_answers, rowvar=False)

dists = pairwise_mahalanobis_distance(voter_answers, candidate_answers, cov)

In [ ]:
def calculate_function_distances(distance_function: Callable, voter_answers: np.ndarray, candidate_answers: np.ndarray,
                                 voter_weights: np.ndarray = None):
    if voter_weights is None:
        voter_weights = np.ones_like(voter_answers)

    voter_answers = voter_answers.reshape(1, -1) if voter_answers.ndim == 1 else voter_answers
    voter_weights = voter_weights.reshape(1, -1) if voter_weights.ndim == 1 else voter_weights
    candidate_answers = candidate_answers.reshape(1, -1) if candidate_answers.ndim == 1 else candidate_answers

    distances = []
    for voter_answer, voter_weight in zip(voter_answers, voter_weights):
        for candidate_answer in candidate_answers:
            distances.append(distance_function(voter_answer, candidate_answer, voter_weight))

    distances = np.array(distances).reshape(len(voter_answers), -1)
    
    return distances

In [ ]:
dist_f = lambda va, ca, vw: scipy.spatial.distance.mahalanobis(va, ca, np.linalg.inv(cov))

In [ ]:
res = calculate_function_distances(dist_f, voter_answers, candidate_answers)

In [ ]:
X1 = voter_answers
X2 = candidate_answers
covariance_matrix = cov

# Step 1: Compute the inverse of the covariance matrix
inv_covariance_matrix = np.linalg.inv(covariance_matrix)

# Step 2: Compute the pairwise differences between the rows of the two matrices
pairwise_diff = X1[:, np.newaxis, :] - X2[np.newaxis, :, :]

# Step 3: Compute the Mahalanobis distance for each pairwise difference
mahalanobis_distance = np.sqrt(np.sum(np.matmul(pairwise_diff, inv_covariance_matrix) * pairwise_diff, axis=-1))

In [ ]:
a = np.diag(np.arange(10))
a

In [ ]:
def pairwise_mahalanobis_distance(X1, X2, covariance_matrix):
    # Step 1: Compute the inverse of the covariance matrix
    inv_covariance_matrix = np.linalg.inv(covariance_matrix)
    
    # Step 2: Compute the pairwise differences between the rows of the two matrices
    pairwise_diff = X1[:, np.newaxis, :] - X2[np.newaxis, :, :]
    
    # Step 3: Compute the Mahalanobis distance for each pairwise difference
    mahalanobis_distance = np.sqrt(np.sum(np.matmul(pairwise_diff, inv_covariance_matrix) * pairwise_diff, axis=-1))
    
    return mahalanobis_distance

In [ ]:
import numpy as np

def pairwise_mahalanobis_distance(X1, X2, covariance_matrix):
    # Step 1: Compute the inverse of the covariance matrix
    inv_covariance_matrix = np.linalg.inv(covariance_matrix)
    
    # Step 2: Compute the pairwise differences between the rows of the two matrices
    pairwise_diff = X1[:, np.newaxis, :] - X2[np.newaxis, :, :]
    
    # Step 3: Compute the Mahalanobis distance for each pairwise difference
    mahalanobis_distance = np.sqrt(np.sum(np.matmul(pairwise_diff, inv_covariance_matrix) * pairwise_diff, axis=-1))
    
    return mahalanobis_distance

# Example usage
X1 = np.array([[1, 2], [3, 4], [5, 6]])
X2 = np.array([[2, 3], [4, 5]])
covariance_matrix = np.eye(2)  # Identity matrix for illustration
distances = pairwise_mahalanobis_distance(X1, X2, covariance_matrix)
print(distances)


In [ ]:
scipy.spatial.distance.pdist()

In [ ]:
px.histogram(np.array([scipy.spatial.distance.euclidean(s1, s2) for s1 in samples2 for s2 in samples2]).reshape(len(samples), -1).ravel())

In [ ]:
px.histogram(np.array([scipy.spatial.distance.euclidean(s1, s2) for s1 in samples for s2 in samples]).reshape(len(samples), -1).ravel())

In [ ]:
res = scipy.spatial.distance.pdist(candidate_answers, metric='mahalanobis')
distances = scipy.spatial.distance.squareform(res)

In [ ]:
# Calculate the Mahalanobis distance between all pairs of respondents
mahalanobis_distances = []
for i in range(len(candidate_answers)):
    for j in range(i+1, len(candidate_answers)):
        distance = mahalanobis(candidate_answers[i], candidate_answers[j], inverse_covariance_matrix)
        mahalanobis_distances.append(distance)

## Minkowski Distance

## Distance Matrices

In [ ]:
ANSWER_POSSIBILITIES

In [ ]:
UNIQUE_DISTANCE_METHODS

In [ ]:
L2_DIST_MAT[RC7[:, None], RC7].sum(axis=0)

In [ ]:
from rsfp.constants import *

In [ ]:
RC = RC5
ic(-L2_DIST_MAT[RC[:, None], RC])
ic(L1_DIST_MAT[RC[:, None], RC])
ic(DIRECTIONAL_DIST_MAT[RC[:, None], RC])
ic(HYBRID_DIST_MAT[RC[:, None], RC])

In [ ]:
question_type = 5
dist_mat = L1_DIST_MAT_UNSCALED[:, ROW_COL_INDICES[question_type]][ROW_COL_INDICES[question_type]].astype(float)
dist_mat -= np.diag(dist_mat.sum(axis=0) - np.min(dist_mat.sum(axis=0)))

In [ ]:
dist_mat


In [ ]:
np.set_printoptions(linewidth=100)
(DIRECTIONAL_DIST_MAT+1).min(axis=1)

## Distance Method Visualization

In [ ]:
def map_points_to_grid(points, grid):
    """
    Map 2D points to the closest point in a specified grid.

    Parameters:
    - points: numpy array of shape (N, 2) containing the coordinates of the points
    - grid: numpy array of shape (M, 2) containing the coordinates of the grid points

    Returns:
    - closest_indices: numpy array of shape (N,) containing the index of the closest grid point for each point
    """

    # Calculate the pairwise Euclidean distances between points and grid points
    distances = np.sqrt(np.sum((points[:, None] - grid) ** 2, axis=2))

    # Find the index of the closest grid point for each point
    closest_indices = np.argmin(distances, axis=1)

    return closest_indices

points = np.stack(np.meshgrid(ANSWER_POSSIBILITIES, ANSWER_POSSIBILITIES), axis=-1).reshape(-1, 2)
mean = df_candidates.a().iloc[:, :2].mean().values
cov = np.cov(df_candidates.a().iloc[:, :2].values, rowvar=False)
samples = np.random.multivariate_normal(mean, cov/2, size=10000)
counts_array = np.zeros(len(points))
for idx, count in zip(*np.unique(map_points_to_grid(samples, points), return_counts=True)):
    counts_array[idx] = count

counts_array

df_samples = pd.DataFrame({'x': points[:, 0], 'y': points[:, 1], 'n_samples': counts_array})
print(cov)
fig = px.scatter(df_samples, x='x', y='y', size='n_samples', height=1000, width=1200)

fig.add_trace(go.Scatter(
        x=[origin_point[0]],
        y=[origin_point[1]],
        mode='markers',
        marker=dict(
            size=30,
            color='red',
            opacity=1,
            symbol='circle'
        ),
        name='Candidates 1'
    ))

#fig.add_traces(px.scatter(df_display, x='x', y='y', color='distance', height=1000, width=DEFAULT_WIDTH, opacity=1, title=f'{distance_method} Distance from Origin', color_continuous_scale='inferno').update_traces(marker=dict(size=20)).data)

In [ ]:
distance_method = 'DM_L1_BONUS'
resolution = 100
x_values = np.linspace(0, 100, resolution)
y_values = np.linspace(0, 100, resolution)
points = np.stack(np.meshgrid(x_values, y_values), axis=-1).reshape(-1, 2)
origin_point = np.array([75, 75])
origin_weights = np.array([2, 1])
max_dist = np.sqrt(((100*origin_weights)**2).sum())
# use actual answer possibilities for grid
points = np.stack(np.meshgrid(ANSWER_POSSIBILITIES, ANSWER_POSSIBILITIES), axis=-1).reshape(-1, 2)

rng = np.random.default_rng(seed=0)
covariance_matrix = np.array(
    [
        [160, -150], 
        [-150, 160]
    ]
)
samples = rng.multivariate_normal(mean=[50, 50], cov=covariance_matrix, size=300)

#distances = np.array([scipy.spatial.distance.mahalanobis(np.zeros(2), point, VI=np.linalg.inv(np.cov(samples, rowvar=False))) for point in points])
#distances = np.array([calculate_distances(np.zeros_like(point), point, distance_method=distance_method, inv_covariance_matrix=np.linalg.inv(np.cov(samples, rowvar=False))) for point in points]).ravel()
distances = np.array([calculate_distances(origin_point, point, origin_weights, distance_method=distance_method, inv_covariance_matrix=np.linalg.inv(np.cov(samples, rowvar=False)), question_type2idx={4: [], 5: [], 7: [0, 1]}) for point in points]).ravel()

df_display = pd.DataFrame({'x': points[:, 0], 'y': points[:, 1], 'Distance': distances})
df_display['Matching'] = 100*(1-(df_display['Distance']/max_dist))
df_display_sample = pd.DataFrame({'x': samples[:, 0], 'y': samples[:, 1]})

marker_size = 20
# add grid points and color them by their distance from origin_point
fig = px.scatter(df_display, x='x', y='y', color='Distance', opacity=0.4, title=f'{distance_method} Distances of Voter (Answers: {origin_point.tolist()} & Weights: {origin_weights.tolist()})', color_continuous_scale='viridis').update_traces(marker=dict(size=marker_size))

# add samples
#fig.add_traces(px.scatter(df_display_sample, x='x', y='y', color_discrete_sequence=['black']).data)

# add origin_point as red marker
fig.add_trace(go.Scatter(
        x=[origin_point[0]],
        y=[origin_point[1]],
        mode='markers',
        marker=dict(
            size=2*marker_size,
            color='red',
            opacity=1,
            symbol='circle'
        ),
        name='Origin Point',
    ))
fig.update_traces(showlegend=False)

fig.update_layout(
    xaxis_title='Answer 1', 
    yaxis_title='Answer 2', 
    height=800, 
    width=800,
    #title='L1 (Manhattan) Distances',
    template='ggplot2',
    plot_bgcolor='white',
    font=dict(
            family="Helvetica",  # Set the font family (optional)
            size=14,        # Set the font size for all text elements
            color="black"   # Set the font color (optional)
        )
)
fig.show()

In [ ]:
distance_method = 'AC'
resolution = 21
x_values = np.linspace(0, 100, resolution)
y_values = np.linspace(0, 100, resolution)
points = np.stack(np.meshgrid(x_values, y_values), axis=-1).reshape(-1, 2)
origin_point = np.array([75, 75])
origin_weights = np.array([1, 1])
max_dist = np.sqrt(((100*origin_weights)**2).sum())
# use actual answer possibilities for grid
#points = np.stack(np.meshgrid(ANSWER_POSSIBILITIES, ANSWER_POSSIBILITIES), axis=-1).reshape(-1, 2)

rng = np.random.default_rng(seed=0)
covariance_matrix = np.array(
    [
        [50, 100], 
        [100, 50]
    ]
)
samples = rng.multivariate_normal(mean=[50, 50], cov=covariance_matrix, size=300)

#distances = np.array([scipy.spatial.distance.mahalanobis(np.zeros(2), point, VI=np.linalg.inv(np.cov(samples, rowvar=False))) for point in points])
#distances = np.array([calculate_distances(np.zeros_like(point), point, distance_method=distance_method, inv_covariance_matrix=np.linalg.inv(np.cov(samples, rowvar=False))) for point in points]).ravel()
distances = np.array([calculate_distances(origin_point, point, origin_weights, distance_method=distance_method, inv_covariance_matrix=np.linalg.inv(np.cov(samples, rowvar=False))) for point in points]).ravel()

df_display = pd.DataFrame({'x': points[:, 0], 'y': points[:, 1], 'Distance': distances})
df_display['Matching'] = 100*(1-(df_display['Distance']/max_dist))
df_display_sample = pd.DataFrame({'x': samples[:, 0], 'y': samples[:, 1]})

marker_size = 8
# add grid points and color them by their distance from origin_point
fig = px.scatter(df_display, x='x', y='y', color='Distance', opacity=0.4, title=f'{distance_method} Distances of Voter (Answers: {origin_point.tolist()} & Weights: {origin_weights.tolist()})', color_continuous_scale='Blues').update_traces(marker=dict(size=marker_size))

# add samples
#fig.add_traces(px.scatter(df_display_sample, x='x', y='y', color_discrete_sequence=['black']).data)

# add origin_point as red marker
# Add outer circle
fig.add_trace(go.Scatter(
    x=[origin_point[0]],
    y=[origin_point[1]],
    mode='markers',
    marker=dict(
        size=2*marker_size,
        color='black',
        opacity=1,
        symbol='circle-open',  # Open circle to mimic the donut's outer ring
        line=dict(width=2)  # Control the thickness of the ring
    ),
    name='Donut Outer Circle',
    showlegend=False
))


fig.update_traces(showlegend=False)

fig.update_layout(
    xaxis_title='Answer 1', 
    yaxis_title='Answer 2', 
    height=600, 
    width=600,
    title='Agreement Count Distances',
    template='ggplot2',
    plot_bgcolor='white',
    font=dict(
            family="Helvetica",  # Set the font family (optional)
            size=14,        # Set the font size for all text elements
            color="black"   # Set the font color (optional)
        )
)
fig.show()

In [ ]:
# Add contour lines
fig= go.Figure(go.Contour(
    z=df_display['Distance'].values.reshape(-1, len(df_display['x'].unique())),
    x=df_display['x'].unique(),
    y=df_display['y'].unique(),
    colorscale='Blues',
    contours=dict(coloring='heatmap'),
    showscale=False,
))

if distance_method in ['mahalanobis_unweighted']:
    # plot candidate samples
    fig.add_traces(px.scatter(df_display_sample, x='x', y='y', color_discrete_sequence=['grey']).update_traces(marker_size=3).data)

# add origin_point
fig.add_trace(go.Scatter(
        x=[origin_point[0]],
        y=[origin_point[1]],
        mode='markers',
        marker=dict(
            size=7,
            color='black',
            opacity=1,
            symbol='x',
        ),
        name='Origin Point',
        showlegend=False,
    ))

# Update layout with a fixed aspect ratio of 1:1 and the specified width
fig.update_layout(
    xaxis_title='Answer 1', 
    yaxis_title='Answer 2',
    xaxis=dict(
        scaleanchor="y",
        scaleratio=1,
        tickvals=[0, 25, 50, 75, 100],  # Only show ticks at these values
    ),
    yaxis=dict(
        scaleanchor="x",
        scaleratio=1,
        tickvals=[0, 25, 50, 75, 100],  # Only show ticks at these values
    ),
    legend=dict(
        title='Distance',  # Add legend title
    ),
    width=figure_width,  # Set the width
    autosize=False,      # Disable autosizing
    height=figure_width,  # Set the height to be the same as the width for a 1:1 aspect ratio
    title='L2 (Smartvote)',
    template='ggplot2',
    plot_bgcolor='white',
    font=dict(
            family="Computer Modern",  # Set the font family (optional)
            size=18,        # Set the font size for all text elements
            color="black"   # Set the font color (optional)
        )
)

fig.show()

In [ ]:
# Define the desired width of the figure
figure_width = 400  # For example, you can change this value as needed

# Add contour lines
fig= go.Figure(go.Contour(
    z=df_display['Distance'].values.reshape(-1, len(df_display['x'].unique())),
    x=df_display['x'].unique(),
    y=df_display['y'].unique(),
    colorscale='Blues',
    contours=dict(coloring='heatmap'),
    showscale=False,
))

# Add samples
#fig.add_traces(px.scatter(df_display_sample, x='x', y='y', color_discrete_sequence=['grey']).update_traces(marker_size=3).data)

# Add origin_point as red marker
fig.add_trace(go.Scatter(
        x=[origin_point[0]],
        y=[origin_point[1]],
        mode='markers',
        marker=dict(
            size=20,
            color='black',
            opacity=1,
            symbol='circle-open'
        ),
        name='Origin Point',
        showlegend=False,
    ))

# Update layout with a fixed aspect ratio of 1:1 and the specified width
fig.update_layout(
    xaxis_title='Answer 1', 
    yaxis_title='Answer 2',
    xaxis=dict(
        scaleanchor="y",
        scaleratio=1,
        tickvals=[0, 25, 50, 75, 100],  # Only show ticks at these values
    ),
    yaxis=dict(
        scaleanchor="x",
        scaleratio=1,
        tickvals=[0, 25, 50, 75, 100],  # Only show ticks at these values
    ),
    legend=dict(
        title='Distance',  # Add legend title
    ),
    width=figure_width,  # Set the width
    autosize=False,      # Disable autosizing
    height=figure_width,  # Set the height to be the same as the width for a 1:1 aspect ratio
    title='L2 (Smartvote)',
    template='ggplot2',
    plot_bgcolor='white',
    font=dict(
            family="Computer Modern",  # Set the font family (optional)
            size=18,        # Set the font size for all text elements
            color="black"   # Set the font color (optional)
        )
)

fig.show()

In [ ]:
save_paper_figure(fig, 'mahalanobis_distance_visualization_computer_modern', height=400, width=400, font_size=16, font_family='Computer Modern', margin_t=10)

In [ ]:
import plotly.graph_objs as go
import numpy as np

# Example 100x100 matrix
matrix = np.random.rand(100, 100)

# Create the heatmap
fig = go.Figure(data=go.Heatmap(
    z=matrix,
    colorscale='Viridis'
))

# Update layout
fig.update_layout(
    title='Heatmap of 100x100 Matrix',
    xaxis_title='X Axis',
    yaxis_title='Y Axis'
)

# Show the figure
fig.show()

In [ ]:
#figure_width = 400
figure_width = 600


for distance_method in [m for m in EVAL_DISTANCE_METHODS if 'DM' not in m]:

    # CALCULATE

    # define grid
    resolution = 101 if distance_method not in ['AC'] else 21
    x_values = np.linspace(0, 100, resolution)
    y_values = np.linspace(0, 100, resolution)
    points = np.stack(np.meshgrid(x_values, y_values), axis=-1).reshape(-1, 2)

    # define origin
    origin_point = np.array([50, 50]) if distance_method in ['mahalanobis_unweighted'] else np.array([75, 75])
    origin_weights = np.array([1, 1])
    max_dist = np.sqrt(((100*origin_weights)**2).sum())

    # compute sample candidates for mahalanobis
    rng = np.random.default_rng(seed=0)
    covariance_matrix = np.array(
        [
            [50, 100], 
            [100, 50]
        ]
    )
    samples = rng.multivariate_normal(mean=[50, 50], cov=covariance_matrix, size=300)

    # compute distances from origin to grid points
    distances = np.array([calculate_distances(origin_point, point, origin_weights, distance_method=distance_method, inv_covariance_matrix=np.linalg.inv(np.cov(samples, rowvar=False))) for point in points]).ravel()

    # define plotting dataframe
    df_display = pd.DataFrame({'x': points[:, 0], 'y': points[:, 1], 'Distance': distances})
    df_display['Matching'] = 100*(1-(df_display['Distance']/max_dist))
    df_display_sample = pd.DataFrame({'x': samples[:, 0], 'y': samples[:, 1]})

    # PLOT

    # Add contour lines
    fig= go.Figure(go.Contour(
        z=df_display['Distance'].values.reshape(-1, len(df_display['x'].unique())),
        x=df_display['x'].unique(),
        y=df_display['y'].unique(),
        colorscale='Blues',
        #colorscale='Oranges',
        contours=dict(coloring='heatmap'),
        showscale=False,
    ))

    if distance_method in ['mahalanobis_unweighted']:
        # plot candidate samples
        fig.add_traces(px.scatter(df_display_sample, x='x', y='y', color_discrete_sequence=['grey']).update_traces(marker_size=3).data)
    
    # add origin_point
    fig.add_trace(go.Scatter(
            x=[origin_point[0]],
            y=[origin_point[1]],
            mode='markers',
            marker=dict(
                size=15,
                color='black',
                opacity=1,
                symbol='x',
            ),
            name='Origin Point',
            showlegend=False,
        ))
    
    # Update layout with a fixed aspect ratio of 1:1 and the specified width
    fig.update_layout(
        xaxis_title='Answer 1', 
        yaxis_title='Answer 2',
        xaxis=dict(
            scaleanchor="y",
            scaleratio=1,
            tickvals=[0, 25, 50, 75, 100],  # Only show ticks at these values
        ),
        yaxis=dict(
            scaleanchor="x",
            scaleratio=1,
            tickvals=[0, 25, 50, 75, 100],  # Only show ticks at these values
        ),
        legend=dict(
            title='Distance',  # Add legend title
        ),
        width=figure_width,  # Set the width
        autosize=False,      # Disable autosizing
        height=figure_width,  # Set the height to be the same as the width for a 1:1 aspect ratio
        title=None,
        template='ggplot2',
        plot_bgcolor='white',
        font=dict(
                family="Times",  # Set the font family (optional)
                size=18,        # Set the font size for all text elements
                color="black"   # Set the font color (optional)
            )
    )

    fig.show()

    save_paper_figure(fig, f'{distance_method}_distance_visualization_times_blues_report', height=figure_width, width=figure_width, font_size=22, font_family='Times', margin_t=10, margin_r=20)

In [ ]:
# Add contour lines
fig= go.Figure(go.Contour(
    z=df_display['Distance'].values.reshape(-1, len(df_display['x'].unique())),
    x=df_display['x'].unique(),
    y=df_display['y'].unique(),
    colorscale='Blues',
    contours=dict(coloring='heatmap')
))

# add samples
fig.add_traces(px.scatter(df_display_sample, x='x', y='y', color_discrete_sequence=['grey']).update_traces(marker_size=3).data)

fig.update_layout(
    xaxis_title='Answer 1', 
    yaxis_title='Answer 2',
    legend=dict(
        title={'text': 'Distance Hello'}  # Add legend title
    ),
    height=600, 
    width=600,
    title='Mahalanobis',
    template='ggplot2',
    plot_bgcolor='white',
    font=dict(
            family="Helvetica",  # Set the font family (optional)
            size=14,        # Set the font size for all text elements
            color="black"   # Set the font color (optional)
        )
)
fig.show()

In [ ]:
save_paper_figure(fig, 'mahalanobis_paper_extra_small', width=360, height=300, margin_b=70, margin_t=20)

In [ ]:
pio.write_image(fig, 'images/L2_sv_small_weight_diff.png', scale=2)

In [ ]:
pio.write_image(fig, 'images/mahalanobis_distance_visualization_contour_title_blues.png', scale=2)

In [ ]:
df_candidates[df_candidates['_answer_strength']<30]['_n_recommendations_spc_L2_normalized'].mean()

In [ ]:
scipy.spatial.distance.euclidean([75]*10, [0]*10)

In [ ]:
df_voters.a()

In [ ]:
df_voters.iloc[0]

In [ ]:
dists = calculate_distances(df_voters.a().iloc[0].values, df_candidates.a().values, df_voters.w().iloc[0].values, distance_method='L2')

In [ ]:
df_c_distance_method_visualization = df_candidates.copy()
df_c_distance_method_visualization['distance'] = dists.ravel()

In [ ]:
df_voters

In [ ]:
dimred = 'PCA'
df_voters['_row_idx'] = df_voters.index
px.scatter(df_voters.sample(1000), x=f'{dimred}_0', y=f'-{dimred}_1', color='_ListParty_1_mahalanobis_unweighted', height=800, hover_data={'_row_idx': True}, width=DEFAULT_WIDTH)

In [ ]:
dimred = 'CA'
marker_size = 3
method = 'mahalanobis_unweighted'
#method = 'DM_L1_BONUS'

df_v = df_voters[(df_voters['_district']=='ZH')][[f'{dimred}_0', f'-{dimred}_1', f'_ListPartyID_1_{method}']].rename(columns={f'_ListPartyID_1_{method}': 'party'})
df_c = df_candidates[df_candidates['_district']=='ZH'][[f'{dimred}_0', f'-{dimred}_1', '_party']].rename(columns={'_party': 'party'})

In [ ]:
category_order = {'party': sorted(set(df_c['party']) | set(df_v['party']))}
category_order

In [ ]:
fig = px.scatter(df_v.sample(5000), x=f'{dimred}_0', y=f'-{dimred}_1', color='party', color_discrete_sequence=px.colors.qualitative.Alphabet, category_orders=category_order, height=800, width=1200, opacity=0.5, title=f'Voter Top List Party Recommendation in Zürich ({method})').update_traces(marker_size=marker_size)
fig.add_traces(
px.scatter(df_c, x=f'{dimred}_0', y=f'-{dimred}_1', color='party', color_discrete_sequence=px.colors.qualitative.Alphabet, category_orders=category_order).update_traces(marker_size=2*marker_size).data
)
fig.update_layout(xaxis_title='Left-Right', yaxis_title='Liberal-Conservative')
fig.show()

In [ ]:
dimred = 'PCA'
marker_size = 5
distance_method = 'mahalanobis_unweighted'

print("Plot")
fig = px.scatter(df_voters[~df_voters['_party'].isna()].sample(1000), x=f'{dimred}_0', y=f'-{dimred}_1', color=f'_ListParty_1_{distance_method}', hover_data={'_party': True}, height=DEFAULT_HEIGHT, width=1200, opacity=1, title=f'List Party Recommendations for Voters', color_continuous_scale='inferno').update_traces(marker=dict(size=marker_size))

fig.show()

In [ ]:
dimred = 'PCA'
marker_size = 5
voter_idx = 110
#voter_idx = 115916
distance_method = 'mahalanobis_unweighted'

inv_cov_candidates = np.linalg.inv(np.cov(df_candidates.a().values, rowvar=False))
dists = calculate_distances(df_voters.a().loc[voter_idx].values, df_candidates.a().values, df_voters.w().loc[voter_idx].values, distance_method=distance_method, inv_covariance_matrix=inv_cov_candidates, question_type2idx=QUESTION_TYPE2IDX)

df_c_distance_method_visualization = df_candidates.copy()
df_c_distance_method_visualization['distance'] = dists.ravel()
print(np.sort(dists)[0][:10])
print("Plot")
fig = px.scatter(df_c_distance_method_visualization, x=f'{dimred}_0', y=f'-{dimred}_1', color='distance', hover_data={'_party': True}, height=800, width=DEFAULT_WIDTH, opacity=1, title=f'Candidate {distance_method} Distance from Voter', color_continuous_scale='inferno').update_traces(marker=dict(size=marker_size))
fig.add_trace(go.Scatter(
        x=[df_voters.loc[voter_idx, f'{dimred}_0']],
        y=[df_voters.loc[voter_idx, f'-{dimred}_1']],
        mode='markers',
        marker=dict(
            size=2*marker_size,
            color='green',
            opacity=1,
            symbol='circle'
        ),
        name='Voter'
    ))

fig.show()

# Filtering

In [ ]:
def get_disagreement_counts(df_voters, df_candidates, skip_neutral_voter_answers: bool = True, voter_map = None, candidate_map = None):

    candidate_map = candidate_map if candidate_map is not None else lambda x: 100 if x > 50 else 0 if x < 50 else 50
    voter_map = voter_map if voter_map is not None else lambda x: 100 if x > 50 else 0 if x < 50 else 50

    candidate_extreme_answers = df_candidates.a().map(candidate_map if callable(candidate_map) else candidate_map.get).to_numpy()
    voter_extreme_answers = df_voters.a().map(voter_map if callable(voter_map) else voter_map.get).to_numpy()
    voter_weights = df_voters.w().to_numpy()

    disagreement_matrix = []
    for voter_extreme_answer, voter_weight in zip(voter_extreme_answers, voter_weights):
        strongly_weighted_idx = np.where((voter_extreme_answer != 50) & (voter_weight == 2))[0] if skip_neutral_voter_answers else np.where(voter_weight == 2)[0]
        disagreement_matrix.append((candidate_extreme_answers[:, strongly_weighted_idx] != voter_extreme_answer[strongly_weighted_idx]).sum(axis=1).reshape(1, -1))

    disagreement_matrix = np.vstack(disagreement_matrix)

    return disagreement_matrix


def add_candidate_recommendations(
        df_voters: SVDataFrame,
        df_candidates: SVDataFrame,
        distance_method: str = 'L2',
        disagreement_counts_kwargs: dict[str, bool] = None,
        inv_covariance_matrix: np.ndarray = None,
        question_type2idx: dict[int, list[int]] = None,
        n_recommendations: int | dict[str, int] = None,
        district_col: str = '_district',
        progress_bar: bool = True
):
    """
    Add candidate recommendations to voter dataframe based on distances computed using specified distance method.

    Parameters
    ----------
    df_voters : SVDataFrame
        DataFrame containing voter data.
    df_candidates : SVDataFrame
        DataFrame containing candidate data.
    distance_method : str, optional
        Distance method to use for computing distances, by default 'L2'.
    inv_covariance_matrix : np.ndarray, optional
        Inverse of covariance matrix for Mahalanobis distance calculation, by default None.
    question_type2idx : dict[int, list[int]], optional
        Dictionary mapping question types to the indices of the corresponding questions in the answer array, by default None.
    n_recommendations : int | dict[str, int], optional
        Number of recommendations to generate for each voter. Can be an integer or a dictionary specifying
        the number of recommendations per district, by default None.
    district_col : str, optional
        Name of the column in the dataframes specifying the district, by default '_district'.
    progress_bar : bool, optional
        Whether to show a progress bar, by default False.

    Returns
    -------
    SVDataFrame
        DataFrame with candidate recommendations added.
    """
    assert df_voters.term == df_candidates.term, "Voter and candidate dataframes must be from the same term."

    if distance_method == 'mahalanobis_unweighted':
        inv_covariance_matrix = np.linalg.inv(
            np.cov(df_candidates.a().values, rowvar=False)) if inv_covariance_matrix is None else inv_covariance_matrix
    if distance_method == 'DM_L1_BONUS':
        question_type2idx = (
            QUESTION_TYPE2IDX if df_voters.term == 2023 else QUESTION_TYPE2IDX19) if question_type2idx is None else question_type2idx
    n_recommendations = (SEATS_PER_CANTON if df_voters.term == 2023 else SEATS_PER_CANTON19) if n_recommendations is None else n_recommendations

    df_voters = df_voters.copy()

    iterator = df_voters[district_col].unique() if not progress_bar else tqdm(df_voters[district_col].unique())
    for district in iterator:
        voter_district_mask = df_voters['_district'] == district
        candidate_district_mask = df_candidates['_district'] == district

        # get the number of recommendations for the current district
        n_recs = min(
            n_recommendations if isinstance(n_recommendations, int) else n_recommendations[district],
            candidate_district_mask.sum()
        )

        # calculate pairwise distances between all voters and candidates
        distances = get_distances(
            df_voters[voter_district_mask],
            df_candidates[candidate_district_mask],
            distance_method=distance_method,
            inv_covariance_matrix=inv_covariance_matrix,
            question_type2idx=question_type2idx,
            progress_bar=progress_bar
        )

        if disagreement_counts_kwargs is not None:
            # calculate pairwise disagreement counts between all voters and candidates
            disagreement_counts = get_disagreement_counts(
                df_voters[voter_district_mask],
                df_candidates[candidate_district_mask],
                **disagreement_counts_kwargs
            )
            # get indices of top recommendations
            # modify distances so that candidates with less disagreements are always preferred, even if they are more distant
            rec_candidates_idx = np.argsort(distances + (distances.max() - distances.min()) * disagreement_counts, axis=1)[:, :n_recs]
        else:
            # get indices of top recommendations
            rec_candidates_idx = np.argsort(distances, axis=1)[:, :n_recs]
            
        # get map from candidate index to candidate ID
        candidate_idx_to_id = df_candidates.loc[candidate_district_mask, 'ID_candidate'].values

        # assign recommendation IDs and distances to voters
        df_voters.loc[voter_district_mask, [f'_matchID_{i}_{distance_method}' for i in range(1, n_recs + 1)]] = \
            candidate_idx_to_id[rec_candidates_idx]
        df_voters.loc[voter_district_mask, [f'_matchDist_{i}_{distance_method}' for i in range(1, n_recs + 1)]] = \
            distances[np.arange(distances.shape[0])[:, None], rec_candidates_idx]

    return df_voters

In [ ]:
d1 = get_disagreement_counts(df_voters[df_voters['_district']=='ZH'], df_candidates[df_candidates['_district']=='ZH'])
d2 = get_disagreement_counts(df_voters[df_voters['_district']=='ZH'], df_candidates[df_candidates['_district']=='ZH'], voter_map={ao: ao for ao in ANSWER_POSSIBILITIES}, candidate_map={ao: ao for ao in ANSWER_POSSIBILITIES})

In [ ]:
px.histogram((d1).min(axis=1), log_y=True)

In [ ]:
px.histogram((d2).min(axis=1), log_y=True)

In [ ]:
px.histogram((df_voters.w() == 2).sum(axis=1), height=800, width=1000, title='Distribution over Number of Strongly Weighted Answers', log_y=True)

In [ ]:
px.histogram(df_voters.a().values[np.where((df_voters.w().values == 2))], log_y=True, title='Distribution over Agreement Values for Strongly Weighted Answers', histnorm='probability', height=800, width=800)

In [ ]:
px.histogram(np.sum((df_voters.a().values == 50) & (df_voters.w().values == 2), axis=1), log_y=True)

In [ ]:
df_voters_exact_filtering = add_candidate_recommendations(
    df_voters, 
    df_candidates, 
    disagreement_counts_kwargs={
        'skip_neutral_voter_answers': False,
        'voter_map': {ap: ap for ap in ANSWER_POSSIBILITIES},
        'candidate_map': {ap: ap for ap in ANSWER_POSSIBILITIES},
    },
)

In [ ]:
df_voters_strong_filtering = add_candidate_recommendations(
    df_voters, 
    df_candidates, 
    disagreement_counts_kwargs={
        'skip_neutral_voter_answers': False,
        'voter_map': lambda x: 100 if x > 50 else 0 if x < 50 else 50,
        'candidate_map': lambda x: x,
    },
)

In [ ]:
df_voters_basic_filtering = add_candidate_recommendations(
    df_voters, 
    df_candidates, 
    disagreement_counts_kwargs={
        'skip_neutral_voter_answers': True,
        'voter_map': None,
        'candidate_map': None,
    },
)

In [ ]:
df_voters_no_filtering = add_candidate_recommendations(df_voters, df_candidates, disagreement_counts_kwargs=None)

In [ ]:
cols = [c for c in df_voters_no_filtering.columns if '_matchID' in c and 'L2' in c]
filter_cols = [f"{c}_filter" for c in cols]
normal_cols = [f"{c}_normal" for c in cols]
df_comp = df_voters_strong_filtering[['recID']+cols].merge(df_voters_no_filtering[['recID']+cols], on='recID', suffixes=['_filter', '_normal'])

df_comp['overlap'] = df_comp.apply(lambda row: len(set(row[filter_cols]) & set(row[normal_cols])), axis=1)
df_comp['n_recs'] = (~df_comp[filter_cols].isna()).sum(axis=1)
df_comp['overlap_frac'] = df_comp['overlap'] / df_comp['n_recs']

In [ ]:
df_voters = df_voters.drop(columns=['overlap', 'n_recs', 'overlap_frac'], errors='ignore').merge(df_comp.drop(columns=filter_cols+normal_cols), on='recID')

In [ ]:
voters_with_strong_weights_mask = (df_voters.w() == 2).sum(axis=1)>0

In [ ]:
px.histogram(df_voters[voters_with_strong_weights_mask], x='overlap_frac', histnorm='probability', log_y=False, nbins=50, height=DEFAULT_HEIGHT, width=DEFAULT_WIDTH, title='Distribution of Overlap Fractions between Candidate L2 Recommendations with and without Basic Filtering')

In [ ]:
px.histogram(df_voters[voters_with_strong_weights_mask], x='overlap_frac', histnorm='probability', log_y=False, nbins=50, height=DEFAULT_HEIGHT, width=DEFAULT_WIDTH, title='Distribution of Overlap Fractions between Candidate L2 Recommendations with and without Exact Filtering')

In [ ]:
px.histogram(df_voters[voters_with_strong_weights_mask], x='overlap_frac', histnorm='probability', log_y=False, nbins=50, height=DEFAULT_HEIGHT, width=DEFAULT_WIDTH, title='Distribution of Overlap Fractions between Candidate L2 Recommendations with Basic and Exact Filtering')

In [ ]:
px.histogram(df_voters[voters_with_strong_weights_mask], x='overlap_frac', histnorm='probability', log_y=False, nbins=50, height=DEFAULT_HEIGHT, width=DEFAULT_WIDTH, title='Distribution of Overlap Fractions between Candidate L2 Recommendations with and without Strong Filtering')

In [ ]:
df_voters['n_filtering_conditions'] = ((df_voters.a() != 50).to_numpy() & (df_voters.w() == 2).to_numpy()).sum(axis=1)

In [ ]:
px.box(df_voters[voters_with_strong_weights_mask], x='n_filtering_conditions', y='overlap_frac', height=800, title='Distribution of Overlap Fractions between Candidate L2 Recommendations with and without Basic Filtering')

In [ ]:
df_voters[df_voters['_matchDist_1_L2'] > df_voters['_matchDist_2_L2']]

In [ ]:
df_candidates = df_candidates.add_recommendation_counts(df_voters_strong_filtering, verbose=True)

In [ ]:
plot_answer_strength_recommendations(df_candidates, trendline='rolling')

In [ ]:
df_candidates = df_candidates.add_recommendation_counts(df_voters_basic_filtering, verbose=True)
plot_answer_strength_recommendations(df_candidates, trendline='rolling', title='AS - Rec Correlation with Basic Filtering')

In [ ]:
df_voters[[c for c in df_voters.columns if '_match' in c]]

In [ ]:
disagreements_matrices = {}
distance_matrices = {}
for district in ['BE']:
    ic(district)
    disagreements_matrices[district] = []
    distance_matrices[district] = []

    # select district voters / candidates
    candidates_answers = df_candidates[df_candidates['_district']==district].a().to_numpy()
    candidates_extreme_answers = df_candidates[df_candidates['_district']==district].a().map(lambda x: 100 if x > 50 else 0 if x < 50 else 50).to_numpy()
    voters_answers = df_voters[df_voters['_district']==district].a().to_numpy()
    voters_extreme_answers = df_voters[df_voters['_district']==district].a().map(lambda x: 100 if x > 50 else 0 if x < 50 else 50).to_numpy()
    voters_weights = df_voters[df_voters['_district']==district].w().to_numpy()

    for v_idx, (voter_answers, voter_extreme_answers, voter_weights) in tqdm(enumerate(zip(voters_answers, voters_extreme_answers, voters_weights))):
    
        strongly_weighted_idx = np.where((voter_extreme_answers != 50) & (voter_weights == 2))[0]
        disagreements_matrices[district].append((candidates_extreme_answers[:, strongly_weighted_idx] != voter_extreme_answers[strongly_weighted_idx]).sum(axis=1).reshape(1, -1))
        distance_matrices[district].append(calculate_distances(voter_answers, candidates_answers, voter_weights).reshape(1, -1))
        
    disagreements_matrices[district] = np.vstack(disagreements_matrices[district])
    distance_matrices[district] = np.vstack(distance_matrices[district])

In [ ]:
da = disagreements_matrices['BE']
di = distance_matrices['BE']

In [ ]:
di

In [ ]:
da.sort(axis=1)
da

In [ ]:
da.min(axis=1)

In [ ]:
(di.max() - di.min()) * da

In [ ]:
da[(range(len(di)), np.argsort(di + (di.max(axis=1).reshape(-1, 1) * da), axis=1)[:, 0])]

In [ ]:
stacked_matrices = np.dstack((matrix1, matrix2))


In [ ]:
np.array([(1,2), (3,4), (0, 5)]).argsort(axis=1)

In [ ]:
tuples_matrix = np.apply_along_axis(lambda x: tuple(x), 2, stacked_matrices)
tuples_matrix

In [ ]:
import numpy as np

# Sample matrices
matrix1 = np.array([[5, 2, 3],
                    [3, 8, 7],
                    [5, 1, 4]])

matrix2 = np.array([[9, 6, 5],
                    [4, 2, 3],
                    [9, 7, 8]])

# Step 1: Sort based on the primary matrix
initial_sort_indices = np.argsort(matrix1, axis=1)

# Step 2: Sort rows of the secondary matrix based on the initial sort indices
sorted_matrix2 = np.take_along_axis(matrix2, initial_sort_indices, axis=1)

# Step 3: Sort the rows of initial_sort_indices based on the sorted secondary matrix
# This ensures that ties are resolved based on matrix2
tie_sorted_indices = np.argsort(sorted_matrix2, axis=1)

# Step 4: Get the final sort indices
final_sort_indices = np.take_along_axis(initial_sort_indices, tie_sorted_indices, axis=1)

# Step 5: Sort the original matrix rows using the final sort indices
sorted_matrix1 = np.take_along_axis(matrix1, final_sort_indices, axis=1)
sorted_matrix2 = np.take_along_axis(matrix2, final_sort_indices, axis=1)

print("Sorted matrix1:\n", sorted_matrix1)
print("Sorted matrix2:\n", sorted_matrix2)


In [ ]:
di

In [ ]:
px.histogram(np.hstack([dm.min(axis=1).ravel() for dm in disagreements_matrices.values()]), histnorm='probability', height=DEFAULT_HEIGHT, width=DEFAULT_WIDTH, title='Distribution over Minimum Number of Disagreements with Any Candidate in the Same District', log_y=True)

In [ ]:
sample = np.random.choice(disagreements_matrix.ravel(), 20000, replace=False)

In [ ]:
disagreements_matrix.max(axis=1).max()

In [ ]:
disagreements_matrix.shape

In [ ]:
fraction_filtered = np.array(fraction_filtered)

In [ ]:
(fraction_filtered == 0).mean()

In [ ]:
(fraction_filtered == 0).mean()

In [ ]:
f1 = fraction_filtered

In [ ]:
# Step 1: Sort the data
sorted_data = np.sort(fraction_filtered)

# Step 2: Calculate the CDF values
cdf_values = np.arange(1, len(sorted_data) + 1) / len(sorted_data)

fig = px.line(y=cdf_values[::10], x=sorted_data[::10], height=DEFAULT_HEIGHT, width=DEFAULT_WIDTH, title='Voter CDF of Fraction of Candidates Left After Filtering')
fig.update_layout(yaxis_title='Fraction of Voters', xaxis_title='Fraction of Candidates Left After Filtering')

In [ ]:
px.histogram(fraction_filtered, height=DEFAULT_HEIGHT, width=DEFAULT_WIDTH)

In [ ]:
px.histogram(df_res.sum(axis=1), log_y=True, title='Distribution of N Strongly Weighted Neutral Answers', height=DEFAULT_HEIGHT, width=DEFAULT_WIDTH)